# graph

> The correction overlay's graph I/O: targeted (scale-shaped) reads of a committed
> spine via the graph-storage `query` action, construction of Correction /
> CorrectionSession nodes + CORRECTS / SUPERSEDES / DERIVED_FROM / REVIEWED edges,
> the in-core effective-spine projection (layer-0 + applied corrections), and
> commit through the job queue. Hand-rolled (revolution-1) = direct CR-18 spec
> material; append-only on layer-0 (never update/delete a Segment).

In [ ]:
#| default_exp graph

In [ ]:
#| export
import json
import time
from uuid import uuid4
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.queue import JobQueue, JobStatus
from cjm_context_graph_primitives.locators import locator_from_dict
# Stage 4: the typed query surface (expressions + results; importing the
# result classes IS the host-side wire registration, F8)
from cjm_context_graph_primitives.query import (
    NodeQuery, EdgeQuery, PropertyPredicate, RelationPredicate, SourcePredicate,
    OrderBy, NodeQueryResult, EdgeQueryResult,
)
from cjm_context_graph_primitives.graph import GraphNode

from cjm_transcript_correction_core.models import (
    Correction, CorrectionSession, CorrectionRelations, SpineSegment,
)

In [ ]:
#| export
async def submit_and_wait(
    queue: JobQueue,                  # Started job queue
    instance_id: str,                 # Capability instance to invoke
    *,
    timeout: Optional[float] = None,  # Seconds to wait; None = no limit
    **kwargs,                         # Forwarded to the capability action
) -> Any:  # Completed job result payload
    """Submit one capability job, wait for it, return its result (raise on failure).

    (Restored as its own cell after the stage-2 field_of retirement removed
    the shared cell that bundled both functions — the loop-back harness
    caught the casualty; one-fn-per-cell prevents the recurrence.)
    """
    job_id = await queue.submit(instance_id, **kwargs)
    job = await queue.wait_for_job(job_id, timeout=timeout)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{instance_id} job {job_id} {job.status}: {job.error}")
    return job.result

In [ ]:
#| export
GRAPH_TASK = "graph-storage"  # The graph-storage adapter task (explicit task channel, stage 4)


async def graph_task(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability instance id
    method: str,      # Adapter method (e.g. "query_nodes", "add_nodes")
    **kwargs,         # Typed-method kwargs (wire dicts ok; the in-worker adapter normalizes)
) -> Any:  # Typed task result (wire-decoded host-side)
    """Invoke a graph-storage adapter method through the queue's task channel.

    The explicit task channel (stage 4) replaces `action=` raw dispatch for
    graph ops: typed expressions in, typed results (NodeQueryResult /
    EdgeQueryResult / GraphNode...) out — still queue-routed (D7 telemetry).
    """
    return await submit_and_wait(queue, graph_id, task=GRAPH_TASK, method=method, **kwargs)


In [ ]:
#| export
# Targeted, scale-shaped reads of one document's spine — re-expressed on the
# TYPED query surface (stage 4): the raw storage-coupled SQL this cell carried
# (ledger C2/C3 — `nodes`/`edges` + json_extract + the PART_OF direction) is
# RETIRED; the expression is portable and the per-backend translation lives in
# the graph-storage tool. Server-side filter/order/page/count throughout.

_SPINE_PROJECTION = ["index", "text", "start_time", "end_time", "sources"]  # + structural "id"


def _row_to_spine_segment(row: Dict[str, Any]) -> SpineSegment:  # One projected row -> SpineSegment
    """Build a SpineSegment from a typed-query projected row (first SourceRef carried)."""
    sources = row.get("sources") or []
    src = sources[0] if sources else {}
    idx = row.get("index")
    return SpineSegment(
        id=row["id"], index=int(idx) if idx is not None else -1,
        text=row.get("text") or "", start_time=row.get("start_time"),
        end_time=row.get("end_time"),
        source_locator=(str(locator_from_dict(src["locator"])) if src.get("locator") else None),
        content_hash=src.get("content_hash"),
    )


def _spine_query(
    document_id: str,  # Document node id
    **overrides,       # NodeQuery field overrides (where / count / limit / ...)
) -> NodeQuery:  # The document-spine read
    """Segments PART_OF the document, ordered by index, spine projection."""
    base: Dict[str, Any] = dict(
        label="Segment",
        related=RelationPredicate("PART_OF", node_id=document_id),
        order_by=OrderBy(prop="index"),
        project=list(_SPINE_PROJECTION),
    )
    base.update(overrides)
    return NodeQuery(**base)


async def load_document_segments(
    queue: JobQueue,               # Started job queue
    graph_id: str,                 # Graph-storage capability id
    document_id: str,              # Document node id
    limit: Optional[int] = None,   # Optional page size
    offset: Optional[int] = None,  # Optional page offset
) -> List[SpineSegment]:  # Ordered spine segments (by index)
    """Load a document's Segment spine, ordered by index (typed query surface)."""
    q = _spine_query(document_id, limit=limit, offset=int(offset or 0))
    res = await graph_task(queue, graph_id, "query_nodes", query=q.to_dict())
    return [_row_to_spine_segment(r) for r in (res.rows or [])]


async def load_empty_segments(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    document_id: str,  # Document node id
) -> List[SpineSegment]:  # Only the empty-text segments (server-side filtered)
    """Load ONLY a document's empty-text segments (server-side filter; D14 prune).

    The evidenced OR case (text = '' OR text IS NULL) = TWO server-side-filtered
    queries unioned client-side — compound boolean predicates stay deferred (P8);
    both halves materialize ~10% of the spine, never the whole document.
    """
    r1 = await graph_task(queue, graph_id, "query_nodes", query=_spine_query(
        document_id, where=[PropertyPredicate("text", "eq", "")]).to_dict())
    r2 = await graph_task(queue, graph_id, "query_nodes", query=_spine_query(
        document_id, where=[PropertyPredicate("text", "is_null")]).to_dict())
    segs = [_row_to_spine_segment(r) for r in (r1.rows or []) + (r2.rows or [])]
    segs.sort(key=lambda s: s.index)
    return segs


async def count_document_segments(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    document_id: str,  # Document node id
) -> int:  # Number of Segment nodes PART_OF the document
    """Count a document's segments server-side (typed count mode)."""
    q = _spine_query(document_id, order_by=None, project=None, count=True)
    res = await graph_task(queue, graph_id, "query_nodes", query=q.to_dict())
    return int(res.count or 0)

In [ ]:
#| export
def _edge(
    source_id: str,                            # Origin node id
    target_id: str,                            # Destination node id
    relation_type: str,                        # Edge relation type
    properties: Optional[Dict[str, Any]] = None,  # Edge properties
) -> Dict[str, Any]:  # Edge wire dict
    """Build an edge wire dict."""
    return {"id": str(uuid4()), "source_id": source_id, "target_id": target_id,
            "relation_type": relation_type, "properties": properties or {}}


def build_correction_node(
    correction_type: str,                  # "text_content" | "punctuation" | "grouping"
    session_id: str,                       # Owning session id
    payload: Dict[str, Any],               # Type-specific payload
    actor: str = "human",                  # Actor
    status: str = "applied",               # Lifecycle status
    canonical_form: Optional[str] = None,  # Optional entity key
    rationale: Optional[str] = None,       # Optional note
) -> Correction:  # The Correction overlay node (not yet committed)
    """Construct a Correction overlay node (pure; commit happens separately)."""
    return Correction(
        correction_type=correction_type, session_id=session_id, payload=payload,
        actor=actor, status=status, canonical_form=canonical_form, rationale=rationale,
    )


def build_prune_correction(
    document_id: str,            # Document being corrected
    pruned: List[SpineSegment],  # Empty layer-0 segments to prune
    session_id: str,             # Owning session id
    actor: str = "human",        # Actor
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:  # (correction node dict, edge dicts)
    """Build one batch grouping Correction that prunes empty segments (D14).

    Non-destructive: layer-0 nodes are NOT deleted; the Correction records the
    pruned ids and DERIVED_FROM edges point at each pruned Segment. The effective
    spine drops them at projection time (reversible by superseding this node).
    """
    payload = {
        "operation": "prune_empty",
        "document_id": document_id,
        "pruned_segment_ids": [s.id for s in pruned],
        "pruned_count": len(pruned),
    }
    node = build_correction_node("grouping", session_id, payload, actor=actor).to_graph_node()
    edges = [_edge(node.id, s.id, CorrectionRelations.DERIVED_FROM) for s in pruned]
    return node.to_dict(), edges

In [ ]:
#| export
async def commit_nodes_edges(
    queue: JobQueue,              # Started job queue
    graph_id: str,                # Graph-storage capability id
    nodes: List[Dict[str, Any]],  # Node wire dicts
    edges: List[Dict[str, Any]],  # Edge wire dicts
) -> Dict[str, int]:  # {"nodes": n, "edges": m} created counts
    """Commit nodes then edges via the task channel (queue-routed; decomp D7 telemetry)."""
    nc = ec = 0
    if nodes:
        ids = await graph_task(queue, graph_id, "add_nodes", nodes=nodes)
        nc = len(ids or [])
    if edges:
        ids = await graph_task(queue, graph_id, "add_edges", edges=edges)
        ec = len(ids or [])
    return {"nodes": nc, "edges": ec}


async def start_session(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    scope: List[str],  # Document ids in scope
) -> CorrectionSession:  # The committed CorrectionSession
    """Create + commit a new CorrectionSession node."""
    sess = CorrectionSession(scope=scope)
    await commit_nodes_edges(queue, graph_id, [sess.to_graph_node().to_dict()], [])
    return sess


async def get_session(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # CorrectionSession node id
) -> Optional[Dict[str, Any]]:  # The session node dict, or None
    """Fetch a CorrectionSession node by id (resume/reopen) — typed get, dict shape preserved."""
    node = await graph_task(queue, graph_id, "get_node", node_id=session_id)
    if node is None:
        return None
    return node.to_dict() if isinstance(node, GraphNode) else node


async def set_session_status(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # CorrectionSession node id
    status: str,      # New status ("in_progress" | "completed" | "reopened")
) -> None:
    """Update a session's status + updated_at.

    The ONLY update_node use in this core: a CorrectionSession is OVERLAY metadata
    whose lifecycle is mutable. Layer-0 Segments + Corrections stay append-only.
    """
    await graph_task(queue, graph_id, "update_node", node_id=session_id,
                     properties={"status": status, "updated_at": time.time()})


async def record_review_markers(
    queue: JobQueue,                   # Started job queue
    graph_id: str,                     # Graph-storage capability id
    session_id: str,                   # Owning session id
    decisions: List[Tuple[str, str]],  # (segment_id, decision) pairs
) -> int:  # Number of REVIEWED edges committed
    """Persist per-(session, segment) review markers as REVIEWED edges."""
    edges = [_edge(session_id, seg_id, CorrectionRelations.REVIEWED, {"decision": dec})
             for seg_id, dec in decisions]
    return (await commit_nodes_edges(queue, graph_id, [], edges))["edges"]


async def load_review_markers(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # Owning session id
) -> Dict[str, str]:  # segment_id -> decision for the session
    """Load a session's review markers (typed edge projection over REVIEWED edges)."""
    q = EdgeQuery(source_id=session_id, relation_type=CorrectionRelations.REVIEWED,
                  project=["decision"])
    res = await graph_task(queue, graph_id, "query_edges", query=q.to_dict())
    return {r["target_id"]: r["decision"] for r in (res.rows or [])}

In [ ]:
#| export
def _node_to_correction_dict(
    node: Any,  # GraphNode (typed task result) or its wire dict
) -> Dict[str, Any]:  # Correction properties + "id" (the pre-stage-4 row shape)
    """Flatten a Correction node to its properties dict + id."""
    d = node.to_dict() if isinstance(node, GraphNode) else dict(node)
    props = dict(d.get("properties", {}) or {})
    props["id"] = d["id"]
    return props


async def find_corrections_for_session(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # Owning session id
) -> List[Dict[str, Any]]:  # Correction property dicts for the session
    """List corrections recorded in a session (typed property filter)."""
    q = NodeQuery(label="Correction",
                  where=[PropertyPredicate("session_id", "eq", session_id)])
    res = await graph_task(queue, graph_id, "query_nodes", query=q.to_dict())
    return [_node_to_correction_dict(n) for n in (res.nodes or [])]


async def find_prior_corrections_by_hash(
    queue: JobQueue,    # Started job queue
    graph_id: str,      # Graph-storage capability id
    content_hash: str,  # SourceRef content hash to look up
) -> List[Dict[str, Any]]:  # Corrections whose CORRECTS target carried this hash
    """Cross-transcript correction-cache lookup (targeted; the graph IS the lexicon).

    THE stage-4 promotion landing: the raw two-table JOIN this function carried
    (C-ledger site 6) became ONE typed far-end provenance constraint
    (`RelationPredicate.node_source`, content-hash-primary per CR-19) — the
    hottest review-path read is portable now.
    """
    q = NodeQuery(label="Correction",
                  related=RelationPredicate(
                      CorrectionRelations.CORRECTS,
                      node_source=SourcePredicate(content_hash=content_hash)))
    res = await graph_task(queue, graph_id, "query_nodes", query=q.to_dict())
    return [_node_to_correction_dict(n) for n in (res.nodes or [])]

In [ ]:
#| export
def project_effective_spine(
    segments: List[SpineSegment],       # Ordered layer-0 spine
    corrections: List[Dict[str, Any]],  # Applied correction property dicts
) -> List[SpineSegment]:  # The effective spine after applying corrections
    """Project the effective spine = layer-0 + applied corrections, resolved in-core.

    Revolution-1 supports the operations built this session:
      - grouping/prune_empty: drop pruned segment ids (re-thread is positional).
      - text_content/replace_text: replace a segment's text by id.
    Superseded corrections are skipped. This hand-rolled projection is direct
    CR-18 spec material (the graph-aware layer should eventually own
    "give me the effective view of this spine").
    """
    pruned_ids = set()
    text_overrides: Dict[str, str] = {}
    for c in corrections:
        if c.get("status") == "superseded":
            continue
        ctype = c.get("correction_type")
        payload = c.get("payload", {}) or {}
        if ctype == "grouping" and payload.get("operation") == "prune_empty":
            pruned_ids.update(payload.get("pruned_segment_ids", []))
        elif ctype == "text_content" and payload.get("operation") == "replace_text":
            tid = payload.get("segment_id")
            if tid is not None:
                text_overrides[tid] = payload.get("new_text", "")
    out: List[SpineSegment] = []
    for s in segments:
        if s.id in pruned_ids:
            continue
        if s.id in text_overrides:
            s = SpineSegment(id=s.id, index=s.index, text=text_overrides[s.id],
                             start_time=s.start_time, end_time=s.end_time,
                             source_locator=s.source_locator, content_hash=s.content_hash)
        out.append(s)
    return out

In [ ]:
#| export
def build_text_correction(
    document_id: str,                      # Document the segment belongs to
    segment_id: str,                       # Layer-0 Segment being corrected
    new_text: str,                         # Corrected text
    session_id: str,                       # Owning session id
    old_text: Optional[str] = None,        # Prior effective text (for the record)
    supersedes_id: Optional[str] = None,   # Prior Correction this one replaces (re-edit)
    actor: str = "human",                  # Actor
    canonical_form: Optional[str] = None,  # Optional entity key (cross-transcript matching)
    rationale: Optional[str] = None,       # Optional note
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:  # (correction node dict, edge dicts)
    """Build a text_content Correction + its CORRECTS (+ optional SUPERSEDES) edges.

    Non-destructive: the layer-0 Segment is unchanged; the correction carries the
    new text in its payload + a CORRECTS edge to the segment. A re-edit adds a
    SUPERSEDES edge (new -> prior) so supersession is graph-native + append-only
    (the prior Correction is never mutated; it is excluded from the effective view
    because it is a SUPERSEDES target). Direct CR-18 spec material.
    """
    payload = {"operation": "replace_text", "document_id": document_id,
               "segment_id": segment_id, "new_text": new_text, "old_text": old_text}
    node = build_correction_node("text_content", session_id, payload, actor=actor,
                                 canonical_form=canonical_form, rationale=rationale).to_graph_node()
    edges = [_edge(node.id, segment_id, CorrectionRelations.CORRECTS)]
    if supersedes_id:
        edges.append(_edge(node.id, supersedes_id, CorrectionRelations.SUPERSEDES))
    return node.to_dict(), edges


async def commit_text_correction(
    queue: JobQueue,                       # Started job queue
    graph_id: str,                         # Graph-storage capability id
    document_id: str,                      # Document the segment belongs to
    segment_id: str,                       # Layer-0 Segment being corrected
    new_text: str,                         # Corrected text
    session_id: str,                       # Owning session id
    old_text: Optional[str] = None,        # Prior effective text
    supersedes_id: Optional[str] = None,   # Prior Correction to supersede (re-edit)
    actor: str = "human",                  # Actor
    canonical_form: Optional[str] = None,  # Optional entity key
) -> str:  # The new Correction node id
    """Commit a text_content correction (node + CORRECTS [+ SUPERSEDES]) + a REVIEWED marker."""
    node, edges = build_text_correction(
        document_id, segment_id, new_text, session_id, old_text=old_text,
        supersedes_id=supersedes_id, actor=actor, canonical_form=canonical_form)
    await commit_nodes_edges(queue, graph_id, [node], edges)
    await record_review_markers(queue, graph_id, session_id, [(segment_id, "corrected")])
    return node["id"]

In [ ]:
#| export
def active_corrections(
    corrections: List[Dict[str, Any]],  # Corrections (e.g. from load_document_corrections)
    superseded_ids: set,                # Ids that are SUPERSEDES targets
) -> List[Dict[str, Any]]:  # Only the effective (non-superseded) corrections
    """Filter to the effective correction set (drop SUPERSEDES targets; append-only supersession)."""
    return [c for c in corrections if c.get("id") not in superseded_ids]


async def _superseded_ids(
    queue: JobQueue,            # Started job queue
    graph_id: str,              # Graph-storage capability id
    correction_ids: List[str],  # Candidate correction ids
) -> set:  # Subset that are SUPERSEDES targets
    """Which of the given corrections are SUPERSEDES targets (typed id-list batch)."""
    if not correction_ids:
        return set()
    q = EdgeQuery(relation_type=CorrectionRelations.SUPERSEDES,
                  target_ids=list(correction_ids), project=[])
    res = await graph_task(queue, graph_id, "query_edges", query=q.to_dict())
    return {r["target_id"] for r in (res.rows or [])}


async def load_document_corrections(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    document_id: str,  # Document node id
) -> Tuple[List[Dict[str, Any]], set]:  # (all corrections for the doc, superseded id set)
    """Load every Correction targeting a document (across sessions) + the superseded-id set.

    Document-scoped (corrections carry payload.document_id — a dotted-path typed
    predicate now) so the effective view is a property of the document, not one
    session — the persistence/resume/reopen requirement. Append-only: supersession
    is read from SUPERSEDES edges, never a status mutation.
    """
    q = NodeQuery(label="Correction",
                  where=[PropertyPredicate("payload.document_id", "eq", document_id)])
    res = await graph_task(queue, graph_id, "query_nodes", query=q.to_dict())
    corrections = [_node_to_correction_dict(n) for n in (res.nodes or [])]
    superseded = await _superseded_ids(queue, graph_id, [c["id"] for c in corrections])
    return corrections, superseded


async def find_active_text_corrections_batch(
    queue: JobQueue,         # Started job queue
    graph_id: str,           # Graph-storage capability id
    segment_ids: List[str],  # Segments to look up
) -> Dict[str, Dict[str, Any]]:  # segment_id -> active text correction
    """Active text corrections for MANY segments in TWO round-trips (C17).

    One far-end batch read (Corrections with CORRECTS edges into the id set;
    `RelationPredicate.node_ids`) + one superseded-set read — replacing the
    per-item lookup the review loop paid (a 1,275-item review would have been
    1,275 queries; now 2). Latest non-superseded correction per segment wins.
    """
    if not segment_ids:
        return {}
    q = NodeQuery(label="Correction",
                  where=[PropertyPredicate("correction_type", "eq", "text_content")],
                  related=RelationPredicate(CorrectionRelations.CORRECTS,
                                            node_ids=list(segment_ids)))
    res = await graph_task(queue, graph_id, "query_nodes", query=q.to_dict())
    cands = [_node_to_correction_dict(n) for n in (res.nodes or [])]
    superseded = await _superseded_ids(queue, graph_id, [c["id"] for c in cands])
    out: Dict[str, Dict[str, Any]] = {}
    for c in cands:
        if c["id"] in superseded:
            continue
        sid = (c.get("payload") or {}).get("segment_id")
        if sid is None:
            continue
        prev = out.get(sid)
        if prev is None or c.get("created_at", 0.0) > prev.get("created_at", 0.0):
            out[sid] = c
    return out


async def find_active_text_correction(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    segment_id: str,  # Segment to look up
) -> Optional[Dict[str, Any]]:  # The current non-superseded text correction, or None
    """Single-segment convenience over the batch read (cross-session; latest wins)."""
    found = await find_active_text_corrections_batch(queue, graph_id, [segment_id])
    return found.get(segment_id)

In [ ]:
# graph pure-logic checks (no runtime)
segs = [
    SpineSegment(id="a", index=0, text="hello"),
    SpineSegment(id="b", index=1, text=""),
    SpineSegment(id="c", index=2, text="world"),
]
empties = [s for s in segs if s.is_empty]
node, edges = build_prune_correction("doc1", empties, session_id="s1")
assert node["label"] == "Correction"
assert node["properties"]["payload"]["pruned_segment_ids"] == ["b"]
assert len(edges) == 1 and edges[0]["relation_type"] == "DERIVED_FROM"
assert edges[0]["target_id"] == "b" and edges[0]["source_id"] == node["id"]

eff = project_effective_spine(segs, [node["properties"]])
assert [s.id for s in eff] == ["a", "c"]               # prune drops the empty segment

repl = {"correction_type": "text_content", "status": "applied",
        "payload": {"operation": "replace_text", "segment_id": "a", "new_text": "HELLO"}}
assert project_effective_spine(segs, [repl])[0].text == "HELLO"

sup = dict(node["properties"]); sup["status"] = "superseded"
assert [s.id for s in project_effective_spine(segs, [sup])] == ["a", "b", "c"]  # superseded ignored
print("graph pure checks OK")

graph pure checks OK


In [ ]:
# graph text-correction pure checks (no runtime)
tn, te = build_text_correction("doc1", "segX", "fixed text", session_id="s1", supersedes_id="prevCorr")
assert tn["properties"]["correction_type"] == "text_content"
assert tn["properties"]["payload"]["segment_id"] == "segX"
assert tn["properties"]["payload"]["new_text"] == "fixed text"
assert tn["properties"]["payload"]["document_id"] == "doc1"
assert {e["relation_type"] for e in te} == {"CORRECTS", "SUPERSEDES"}
corr_edge = next(e for e in te if e["relation_type"] == "CORRECTS")
assert corr_edge["target_id"] == "segX" and corr_edge["source_id"] == tn["id"]
assert next(e for e in te if e["relation_type"] == "SUPERSEDES")["target_id"] == "prevCorr"
assert [c["id"] for c in active_corrections([{"id": "a"}, {"id": "b"}, {"id": "c"}], {"b"})] == ["a", "c"]
print("graph text-correction checks OK")